In [1]:
import numpy as np
import pandas as pd
from pylab import *
import seaborn as sns
import pickle
import matplotlib.pyplot as plt
import pandas as pd
from pylab import *
from sequana import FastA
import tensorflow as tf
from tensorflow.keras import layers, models
from tqdm import tqdm
from sklearn.model_selection import train_test_split

import random
from collections import defaultdict
from sklearn.metrics import classification_report


2025-07-28 09:35:58.350256: I external/local_xla/xla/tsl/cuda/cudart_stub.cc:32] Could not find cuda drivers on your machine, GPU will not be used.
2025-07-28 09:35:59.688648: I external/local_xla/xla/tsl/cuda/cudart_stub.cc:32] Could not find cuda drivers on your machine, GPU will not be used.
2025-07-28 09:36:00.349437: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1753688161.014654    4664 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1753688161.170049    4664 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1753688162.463200    4664 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linkin

In [2]:
centromeres = pd.read_csv("../../output/estimation/major/Major.csv")
centromeres = {
    str(row['Chromosome']): (row['start'], row['end'])
    for _, row in centromeres.iterrows()
}


In [3]:
f = FastA("../../data/Fasta/TriTrypDB-68_LmajorFriedlin_Genome.fasta")

In [4]:
def one_hot_encoding(x):
    if x == 'A':
        return np.array([1,0,0,0])
    elif x == 'C':
        return np.array([0,1,0,0])
    elif x == 'G':
        return np.array([0,0,1,0])
    elif x == 'T':
        return np.array([0,0,0,1])
    else:
        return np.array([0,0,0,0])
        

In [5]:
SIZE=3000

In [6]:
def get_true_positive(size=3000):
    data = []
    for chrom in range(1,36+1):
        start, stop = centromeres[str(chrom)]
        #seq = f.sequences[f.names.index(str(chrom))]
        seq = f.sequences[chrom-1]

        if stop-start != 3000:
            stop = start + size
        # flip to get more positives
        data.append([one_hot_encoding(x) for x in seq[start:stop]])
        data.append([one_hot_encoding(x) for x in seq[start:stop][::-1]])
        

    return data
positives = get_true_positive(SIZE)

In [7]:
lengths = list(f.get_lengths_as_dict().values())

In [8]:

##################################### WARNING #############################################
############################### REMOVE FALSE NEGATIVE (centromeres) #######################

def get_true_negatives(N=1000,size=3000,seed=42):
    data = []
    positions = defaultdict(list)
    random.seed(seed)

    # get the random combos first
    for i in tqdm(range(N)):
        chrom = random.randint(1,36)
        N = lengths[chrom-1]
        pos = random.randint(1, N-size)
        start, stop = centromeres[str(chrom)]
        if pos>start and pos<stop:
            pass # this is a centromeres so not a negative
        else:
            positions[chrom].append(pos)
        
        
    for chrom in tqdm(positions.keys()):
        seq = f.sequences[chrom-1]
        for position in positions[chrom]:
            data.append([one_hot_encoding(x) for x in seq[position:position+size]])
    return data

    #negatives = get_true_negatives(N=10000, size=SIZE)

In [17]:
kermel_values = [[13,9],[18,12]]
for kermel in kermel_values:
    results = []
    f = FastA("../../data/Fasta/TriTrypDB-68_LmajorFriedlin_Genome.fasta")
    for i in range(1,10):
        print("###############################################################################")
        print(f'{i}/10')
        print("###############################################################################")

        negatives = get_true_negatives(N=10000, size=SIZE,seed=i)
     
        
        X_pos = positives
        X_neg = negatives
        
        
        # Create label arrays
        y_pos = [1] * len(X_pos)
        y_neg = [0] * len(X_neg)
        
        # Combine and shuffle
        X = np.array(X_pos + X_neg)  # shape: (N, 3000, 4)
        y = np.array(y_pos + y_neg)
        
        X_train, X_test, y_train, y_test = train_test_split(
            X, y, test_size=0.2, stratify=y, random_state=42
        )
    
    
        layer_size=16

        model = models.Sequential([
            layers.Input(shape=(SIZE, 4)),
            layers.Conv1D(layer_size, kernel_size=kermel[0], activation='relu'),
            layers.MaxPooling1D(pool_size=2),
            layers.Conv1D(layer_size*2, kernel_size=kermel[1], activation='relu'),
            layers.GlobalMaxPooling1D(),
            layers.Dense(layer_size, activation='relu'),
            layers.Dense(1, activation='sigmoid')  # binary classification
        ])
        model.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])
        
        seed = 42
    
        tf.random.set_seed(seed)
        
        tf.config.experimental.enable_op_determinism()
        history = model.fit(
            X_train, y_train,
            validation_data=(X_test, y_test),
            epochs=40,
            batch_size=32,
            class_weight={0: 1, 1: len(y_neg)/len(y_pos)},  # handle imbalance
            callbacks=[tf.keras.callbacks.EarlyStopping(patience=3, restore_best_weights=True)]
        )
    
    
        y_pred = model.predict(X_test) > 0.2
        report = classification_report(y_test, y_pred, output_dict=True)
    
        metrics = {
            'f1_score': report['1']['f1-score'],
            'precision': report['1']['precision'],
            'recall': report['1']['recall']
        }
        results.append(metrics)

    with open(f"{kermel[0]}_{kermel[1]}_kermel_result.pkl", "wb") as f:
        pickle.dump(results, f)




###############################################################################
1/10
###############################################################################


100%|███████████████████████████████████████████| 36/36 [00:31<00:00,  1.14it/s]


Epoch 1/40


2025-07-28 11:33:02.395996: E tensorflow/core/framework/node_def_util.cc:680] NodeDef mentions attribute use_unbounded_threadpool which is not in the op definition: Op<name=MapDataset; signature=input_dataset:variant, other_arguments: -> handle:variant; attr=f:func; attr=Targuments:list(type),min=0; attr=output_types:list(type),min=1; attr=output_shapes:list(shape),min=1; attr=use_inter_op_parallelism:bool,default=true; attr=preserve_cardinality:bool,default=false; attr=force_synchronous:bool,default=false; attr=metadata:string,default=""> This may be expected if your graph generating binary is newer  than this binary. Unknown attributes will be ignored. NodeDef: {{node ParallelMapDatasetV2/_16}}


250/251 ━━━━━━━━━━━━━━━━━━━━ 0s 35ms/step - accuracy: 0.7835 - loss: 1.3670

...

250/251 ━━━━━━━━━━━━━━━━━━━━ 0s 59ms/step - accuracy: 0.6279 - loss: 1.4592

2025-07-28 13:35:44.578161: E tensorflow/core/framework/node_def_util.cc:680] NodeDef mentions attribute use_unbounded_threadpool which is not in the op definition: Op<name=MapDataset; signature=input_dataset:variant, other_arguments: -> handle:variant; attr=f:func; attr=Targuments:list(type),min=0; attr=output_types:list(type),min=1; attr=output_shapes:list(shape),min=1; attr=use_inter_op_parallelism:bool,default=true; attr=preserve_cardinality:bool,default=false; attr=force_synchronous:bool,default=false; attr=metadata:string,default=""> This may be expected if your graph generating binary is newer  than this binary. Unknown attributes will be ignored. NodeDef: {{node ParallelMapDatasetV2/_15}}


251/251 ━━━━━━━━━━━━━━━━━━━━ 18s 63ms/step - accuracy: 0.6280 - loss: 1.4588 - val_accuracy: 0.3766 - val_loss: 0.6977
Epoch 2/40
251/251 ━━━━━━━━━━━━━━━━━━━━ 16s 63ms/step - accuracy: 0.4575 - loss: 1.3768 - val_accuracy: 0.0070 - val_loss: 0.8902
Epoch 3/40
251/251 ━━━━━━━━━━━━━━━━━━━━ 15s 61ms/step - accuracy: 0.5542 - loss: 1.2074 - val_accuracy: 0.9610 - val_loss: 0.3303
Epoch 4/40
251/251 ━━━━━━━━━━━━━━━━━━━━ 16s 63ms/step - accuracy: 0.8765 - loss: 0.7478 - val_accuracy: 0.9545 - val_loss: 0.2192
Epoch 5/40
251/251 ━━━━━━━━━━━━━━━━━━━━ 20s 61ms/step - accuracy: 0.9778 - loss: 0.2526 - val_accuracy: 0.9865 - val_loss: 0.0762
Epoch 6/40
251/251 ━━━━━━━━━━━━━━━━━━━━ 16s 63ms/step - accuracy: 0.9902 - loss: 0.0931 - val_accuracy: 0.9870 - val_loss: 0.0578
Epoch 7/40
251/251 ━━━━━━━━━━━━━━━━━━━━ 16s 63ms/step - accuracy: 0.9935 - loss: 0.0489 - val_accuracy: 0.9850 - val_loss: 0.0624
Epoch 8/40
251/251 ━━━━━━━━━━━━━━━━━━━━ 16s 65ms/step - accuracy: 0.9944 - loss: 0.0337 - val_accurac

2025-07-28 13:40:17.001828: E tensorflow/core/framework/node_def_util.cc:680] NodeDef mentions attribute use_unbounded_threadpool which is not in the op definition: Op<name=MapDataset; signature=input_dataset:variant, other_arguments: -> handle:variant; attr=f:func; attr=Targuments:list(type),min=0; attr=output_types:list(type),min=1; attr=output_shapes:list(shape),min=1; attr=use_inter_op_parallelism:bool,default=true; attr=preserve_cardinality:bool,default=false; attr=force_synchronous:bool,default=false; attr=metadata:string,default=""> This may be expected if your graph generating binary is newer  than this binary. Unknown attributes will be ignored. NodeDef: {{node ParallelMapDatasetV2/_14}}


63/63 ━━━━━━━━━━━━━━━━━━━━ 1s 15ms/step


In [11]:
with open(f"{64}_layer_result_patience5.pkl", "wb") as f:
    pickle.dump(results, f)


In [32]:
import pickle
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

# Liste des tailles de couches
layer_sizes = [16, 32, 64, 128]

# Charger les résultats et les mettre dans une liste
all_data = []

for layer in layer_sizes: 
    with open(f"{layer}_layer_result.pkl", "rb") as f:
        results = pickle.load(f)
        for metrics in results:
            all_data.append({
                'layer_size': layer,
                'f1_score': metrics['f1_score'],
                'precision': metrics['precision'],
                'recall': metrics['recall']
            })

# Convertir en DataFrame
df = pd.DataFrame(all_data)

# Afficher les 3 boxplots
metrics = ['f1_score', 'precision', 'recall']
for metric in metrics:
    plt.figure(figsize=(8, 6))
    sns.boxplot(x='layer_size', y=metric, data=df)
    plt.title(f"Boxplot of {metric} by Layer Size")
    plt.xlabel("Layer Size")
    plt.ylabel(metric.capitalize())
    plt.grid(True)
    plt.tight_layout()
    plt.show()


(8011, 2003)